In [4]:
!pip -q install ultralytics scikit-learn pyyaml pandas

In [1]:
#import libraries
import csv
import cv2
from ultralytics import YOLO
import numpy as np
from IPython.display import Video, display
from pathlib import Path

In [ ]:
#Project Path
PROJECT_ROOT = Path("modules/Tracking")
print("Project root:", PROJECT_ROOT)

Project root: ..\modules\Tracking


In [ ]:
RUNS_PATH = PROJECT_ROOT / "runs"
DETECTION_RUNS_PATH = Path("modules/Detection/runs")
#vdo_path = "C:/Users/ElricM/OneDrive - VTI/Videos/Lkpg_bus_terminal.avi_trimmed1.mp4"
part = "4"
vdo_path = f"C:/Users/ElricM/OneDrive - VTI/Videos/Lkpg_bus_terminal_stabilized_part{part}.mp4"
terminal_name = "lkpg"

In [16]:

vdo_path = Path(vdo_path)
runs_path = Path(RUNS_PATH)

# Model path
model_path = DETECTION_RUNS_PATH / "yolov12s4" / "weights" / "best.pt"
model = YOLO(str(model_path))

# Open video
cap = cv2.VideoCapture(str(vdo_path))
if not cap.isOpened():
    raise ValueError(f"Could not open video: {vdo_path}")

# Video properties
fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0:
    fps = 29.97  # fallback

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

frame_idx = 0
unique_track_ids = set()

# Output folder
track_runs_path = runs_path / f"track_{terminal_name}"
track_runs_path.mkdir(parents=True, exist_ok=True)

# Output video path
output_video_path = track_runs_path / f"yolo12s_{terminal_name}_part{part}.mp4"

# Video writer
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(str(output_video_path), fourcc, fps, (width, height))

# CSV output
csv_path = track_runs_path / f"{terminal_name}_full_bbox_centers_xyxy_yolo12s_part{part}.csv"
csv_file = open(str(csv_path), mode="w", newline="", encoding="utf-8")
writer = csv.writer(csv_file)
writer.writerow([
    "frame_idx", "time_sec", "track_id",
    "x1", "y1", "x2", "y2", "x_center", "y_center"
])

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx += 1

    # Track objects
    results = model.track(
        frame,
        persist=True,
        tracker="botsort.yaml",
        conf=0.7,
        iou=0.7,
        verbose=False
    )

    annotated_frame = results[0].plot()

    if results[0].boxes is not None and results[0].boxes.id is not None:
        track_ids = results[0].boxes.id.cpu().numpy()
        boxes_xyxy = results[0].boxes.xyxy.cpu().numpy()
        time_sec = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0

        for track_id, box in zip(track_ids, boxes_xyxy):
            x1, y1, x2, y2 = box
            x_center = (x1 + x2) / 2
            y_center = (y1 + y2) / 2

            writer.writerow([
                frame_idx, time_sec, int(track_id),
                float(x1), float(y1), float(x2), float(y2),
                float(x_center), float(y_center)
            ])

            unique_track_ids.add(int(track_id))

    cv2.putText(
        annotated_frame,
        f"Track IDs: {len(unique_track_ids)}",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    out.write(annotated_frame)

    # Optional preview every 100 frames
    if frame_idx % 100 == 0:
        print(f"Processed frame {frame_idx}")
        # Uncomment to preview frames in Colab:
        # cv2_imshow(annotated_frame)

# Cleanup
cap.release()
out.release()
csv_file.close()

print("Processing complete.")
print(f"Output video saved to: {output_video_path}")
print(f"CSV saved to: {csv_path}")

# Display the saved video in Colab
#display(Video(str(output_video_path), embed=True))

Processed frame 100
Processed frame 200
Processed frame 300
Processed frame 400
Processed frame 500
Processed frame 600
Processed frame 700
Processed frame 800
Processed frame 900
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
Processed frame 1000
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching p